In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

LATEX_ACCENTS = {
    "'": {"a": "á", "e": "é", "i": "í", "o": "ó", "u": "ú", "y": "ý",
          "A": "Á", "E": "É", "I": "Í", "O": "Ó", "U": "Ú", "Y": "Ý"},
    "`": {"a": "à", "e": "è", "i": "ì", "o": "ò", "u": "ù",
          "A": "À", "E": "È", "I": "Ì", "O": "Ò", "U": "Ù"},
    '"': {"a": "ä", "e": "ë", "i": "ï", "o": "ö", "u": "ü", "y": "ÿ",
          "A": "Ä", "E": "Ë", "I": "Ï", "O": "Ö", "U": "Ü"},
    "^": {"a": "â", "e": "ê", "i": "î", "o": "ô", "u": "û",
          "A": "Â", "E": "Ê", "I": "Î", "O": "Ô", "U": "Û"},
    "~": {"a": "ã", "n": "ñ", "o": "õ",
          "A": "Ã", "N": "Ñ", "O": "Õ"},
    "c": {"c": "ç", "C": "Ç"},
    "v": {"s": "š", "z": "ž", "c": "č", "r": "ř", "n": "ň",
          "S": "Š", "Z": "Ž", "C": "Č", "R": "Ř", "N": "Ň"},
}

df_bronze = spark.table("arxivist_ai_dev.bronze.arxiv_metadata")

def _replace_accent(match):
    accent = match.group(1)
    letter = match.group(2)
    return LATEX_ACCENTS.get(accent, {}).get(letter, letter)

def clean_latex(text):
    if text is None:
        return None
    text = re.sub(r"\\(['\"`^~cv])\{([a-zA-Z])\}", _replace_accent, text)
    text = re.sub(r"\\(['\"`^~cv])([a-zA-Z])", _replace_accent, text)
    text = re.sub(r'\\(?:textbf|textit|emph|mathrm|mathbf|mathbb|tt|rm|it|bf)\s*', '', text)
    text = text.replace('\\$', '$')
    text = text.replace('\\%', '%')
    text = text.replace('\\&', '&')
    text = text.replace('\\rightarrow', '→')
    text = text.replace('\\leftarrow', '←')
    text = text.replace('\\approx', '≈')
    text = text.replace('\\leq', '≤')
    text = text.replace('\\geq', '≥')
    text = text.replace('\\times', '×')
    text = text.replace('\\pm', '±')
    text = text.replace('\\infty', '∞')
    text = re.sub(r'\$([^$]*)\$', r'\1', text)
    text = re.sub(r'\\[a-zA-Z]+', '', text)
    text = re.sub(r'[{}]', '', text)
    text = re.sub(r'[\^_]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

clean_latex_udf = F.udf(clean_latex, StringType())

preview = (df_bronze
    .filter(F.col("title").rlike(r"\\['`\"^~]"))
    .select(
        F.col("title").alias("title_raw"),
        clean_latex_udf(F.col("title")).alias("title_cleaned")
    )
    .limit(10)
)
display(preview)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_cs = df_bronze.filter(
    F.forall(F.split(F.col("categories"), " "), lambda x: x.startswith("cs."))
)

df_selected = df_cs.select(
    F.col("id"),                          
    clean_latex_udf(F.col("title")).alias("title"),       
    clean_latex_udf(F.col("abstract")).alias("abstract"),  
    F.col("categories"),                   
    F.col("authors"),                      
    F.col("update_date"),                  
)

w = Window.partitionBy("id").orderBy(F.col("update_date").desc())
df_silver = (
    df_selected
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

print(f"Before dedup: {df_cs.count():,}")
print(f"After dedup:  {df_silver.count():,}")
print(f"Duplicates removed: {df_cs.count() - df_silver.count():,}")
display(df_silver.limit(20))

In [0]:
df_silver.write.mode("overwrite").saveAsTable("arxivist_ai_dev.silver.arxiv_cs_papers")

count = spark.read.table("arxivist_ai_dev.silver.arxiv_cs_papers").count()